# STEREO / SWAVES dynamic spectrum

This notebook plots the combined STEREO-A SWAVES Level-2 product. SWAVES is
a sweep-frequency radio receiver covering **2.6 kHz - 16.025 MHz** in a single
contiguous band.

The relevant CDF variables are:

- `Epoch`: time stamps
- `frequency`: frequency axis in kHz
- `avg_intens_ahead`: STEREO-A combined intensity (use `avg_intens_behind`
  for STEREO-B if needed)

Two loaders are provided: a manual `pycdf` reader (default) and an alternative
based on `pyspedas`.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from matplotlib.ticker import AutoMinorLocator
from matplotlib.colors import LogNorm
import matplotlib as mpl

# Use the precise matplotlib epoch (avoids ~10 us offsets in old matplotlib).
mpl.rcParams['date.epoch'] = '1970-01-01T00:00:00'
try:
    mdates.set_epoch('1970-01-01T00:00:00')
except RuntimeError:
    pass

# Unified plotting style for all dynamic spectra notebooks.
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11

from astropy.visualization import ImageNormalize, PercentileInterval


## Configuration


In [ ]:
mydate = '2024-05-14'
year, month, day = mydate.split('-')

data_dir = './sample_data/swaves'
outputs  = './outputs'
os.makedirs(outputs, exist_ok=True)


## Helper functions


In [ ]:
def subtract_background_median(df):
    """
    Subtract a per-channel median background from a dynamic spectrum.

    The function computes the median along the time axis (axis=0) for each
    frequency channel, then subtracts it from every time sample of that
    channel. This is the standard approach for highlighting transient
    emission against a slowly-varying instrumental/sky background.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame with shape (n_times, n_freqs). Index is time, columns
        are frequencies.

    Returns
    -------
    pandas.DataFrame
        Same shape as input with the per-channel median removed.
    """
    bkg = np.nanmedian(df.values, axis=0)
    return df - np.tile(bkg, (df.shape[0], 1))


## Load STEREO/SWAVES with `pycdf` (default)


In [ ]:
from spacepy import pycdf

swaves_files = sorted(glob.glob(
    f'{data_dir}/stereo_level2_swaves_{year}{month}{day}*.cdf'
))
print(*swaves_files, sep='\n')
cdf = pycdf.CDF(swaves_files[0])

time_ste = np.asarray(cdf['Epoch'])
freq_ste = np.asarray(cdf['frequency'])          # kHz
data_ste = np.asarray(cdf['avg_intens_ahead'])   # STEREO-A

print(f'Shape (time, freq): {data_ste.shape}')
print(f'Time range: {time_ste[0]} -> {time_ste[-1]}')
print(f'Freq range: {freq_ste[0]:.2f} kHz - {freq_ste[-1]/1e3:.2f} MHz')


## Alternative loader: `pyspedas`

Uncomment to use `pyspedas.stereo.waves` instead.


In [ ]:
# import pyspedas
# from pytplot import get_data
#
# pyspedas.stereo.waves(trange=[mydate, mydate], probe='a', datatype='l2_avg', no_update=True)
# ts, val, freq_ste = get_data('avg_intens_ahead')
# time_ste = pd.to_datetime(ts, unit='s').to_pydatetime()
# data_ste = val  # shape (n_time, n_freq), units already in kHz for freq_ste


## Remove the per-channel median background


In [ ]:
df_ste = pd.DataFrame(data_ste, index=time_ste, columns=freq_ste)
df_ste_nobkg = subtract_background_median(df_ste)


## Plot the dynamic spectrum


In [ ]:
# Use a percentile-clipped normalisation; SWAVES dynamic range is large.
norm = ImageNormalize(df_ste_nobkg.values, interval=PercentileInterval(98))

fig = plt.figure(figsize=(12, 5))
ax  = fig.add_subplot(111)
pc  = ax.pcolormesh(
    df_ste_nobkg.index,
    df_ste_nobkg.columns / 1e3,         # convert kHz -> MHz for y-axis label
    df_ste_nobkg.values.T,
    norm=norm, cmap='Spectral_r',
)
fig.colorbar(pc, ax=ax, pad=0.02, label='dB (background subtracted)')

ax.set_yscale('log')
ax.set_ylim(ax.get_ylim()[::-1])
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_title(f'STEREO-A / SWAVES  -  {mydate}'
             f'  ({freq_ste[0]:.2f} kHz - {freq_ste[-1]/1e3:.2f} MHz)')

fig.tight_layout()
fig.savefig(f'{outputs}/swaves_dyspec_{mydate}.png', bbox_inches='tight')
plt.show()
